# 25 CBS-TA：联合目标分配与路径


本课按百科条目写：先定义……调度台在最后，可跳过。

---

## 1. 定义

**CBS-TA**（Conflict-Based Search with Task Assignment，Hönig 等）：同时选择指派 \(\pi\) 和一组 **无碰撞** 路径，优化 SOC（各车到达时间之和），不是只优化矩阵和。

- **输入**：图 G，`robots`，`starts`，`tasks`（与 22 课相同的一批静态任务）。
- **输出**：匹配 `assign`，无碰撞 `paths`，以及两套数字：`assign_cost`（矩阵和）与 `soc`（含等待）。

完整算法在 **指派树** 上枚举匹配（下一最优匹配 / Murty 一类），每个匹配当作 CBS 的一组目标；高层仍按 SOC 最佳优先。本仓库教学实现更小：先匈牙利得 \(\pi_H\)，跑一遍 CBS；再贪心得 \(\pi_G\)，再跑一遍 CBS；报告两者 SOC。n=2 时这两种已经覆盖全部完美匹配。

**关键句：** 「先匈牙利再 CBS」给出的是 **该 \(\pi\) 下** 的 SOC 最优路径，不是所有 \(\pi\) 里 SOC 最小的那个。等待可以让矩阵更优的匹配在时空里更差。

---

## 2. 和谁相邻

|  | 匈牙利 | 匈牙利+CBS | CBS-TA | 贪心+CBS |
|--|--------|------------|--------|----------|
| 决策 | 只匹配 | 匹配固定再消撞 | 匹配与路径一起搜 | 匹配次优再消撞 |
| 目标函数 | \(\sum C_{ij}\) | 该 π 的 SOC | 全局 SOC | 该 π 的 SOC |
| 碰撞 | 不管 | 管 | 管 | 管 |

和 22：矩阵最优 ≠ 时空最优。  
和 15 课 CBS：CBS 的目标是给定的；本课目标本身是搜索变量。  
和 TPTS：TPTS 在线启发式；CBS-TA 离线、小 n、求质量。

---

## 3. 不变量 / 定理

两套代价，不要混：

| 符号 | 定义 | 含等待？ |
|------|------|----------|
| 矩阵和 `assign_cost` | \(\sum_i \mathrm{Dijkstra}(s_i, \pi(i))\) | 否 |
| SOC | \(\sum_i T_i^{\mathrm{arr}}\)（路径时间戳，含 wait） | 是 |

**引理（固定 π）：** CBS 在该目标下返回的 SOC 最优（经典离散 MAPF、stay-at-target 约定与 15 课相同）。

**不是引理：** 矩阵最优的 \(\pi_H\) 的 SOC ≤ 其他 \(\pi\) 的 SOC。反例结构：

|  | T0 | T1 |
|--|----|----|
| r0 | 1 | 5 |
| r1 | 5 | 2 |

匈牙利取 r0–T0、r1–T1，矩阵和 3。若这两条最短路在同一格对撞，CBS 让一车等待 10 拍，SOC=13。另一匹配矩阵和 10，但各走各的互不挡，SOC=10，**SOC 更好**。本课地图更温和：两种匹配矩阵和都是 4，CBS 后 SOC 都是 5（都要抢 J）。

**单射：** \(\pi\) 必须一人一任务。两车同一目标 ⇒ stay-at-target 下终点永远顶点冲突，CBS 会把 cap 用完。

---

## 4. 完整伪代码（与下面 Python 对应）

```
function CompareAssignThenCBS(G, robots, starts, tasks):
    πH, cH ← Hungarian(G, ...)          # 矩阵最优
    πG, cG ← Greedy(G, ...)
    rH ← CBS(G, starts, πH)             # 该 π 的无碰撞路径
    rG ← CBS(G, starts, πG)
    print 矩阵和 cH, cG
    print SOC rH.soc, rG.soc
    print 路径（看哪一拍在等待）
    return rH, rG

function CBS_TA_lib(...):               # 本仓库
    hung ← Hungarian
    planned ← CBS(starts, hung.assign)
    planned.assign_cost ← hung.cost
    planned.greedy_soc ← CBS(greedy.assign).soc
    return planned
```

完整文献 CBS-TA 还要：从匈牙利开始生成下一最优匹配，CBS 高层节点带「当前 π」，直到 SOC 下界证明没有更好 π。n=2 穷举两种匹配已经穷尽。

---

## 5. 复杂度

匹配个数最多 n!。每个匹配一次 CBS，CBS 最坏对冲突数指数。教学 n=2：2 次 CBS。填矩阵仍是 \(n^2\) 次 Dijkstra。所以「先匹配再 CBS」的代价几乎全在 CBS，不在匈牙利。

---

## 6. 完整手算 / 小表

`assign_cross_map`，r0@R0、r1@R1，任务 G0/G1。矩阵全 2，两种匹配矩阵和都是 4。

两人去右侧都必须经过 J。一种无碰撞时间表（库 CBS 实际输出）：

| t | r0 | r1 |
|---|----|----|
| 0 | R0 | R1 |
| 1 | R0（等） | J |
| 2 | J | G1 |
| 3 | G0 | G1 |

SOC = 3+2 = 5。矩阵和 4，多出来的 1 就是 r0 在 R0 的等待。交叉匹配 r0–G1、r1–G0 同样抢 J，SOC 仍是 5。

**数字反例（概念，不必在本图上出现）：** 矩阵最优 3 但等待 +10 ⇒ SOC 13；另一匹配 10、无等待 ⇒ SOC 10。比较算法时必须说清比的是哪一列。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 7. 分步实现（自己写，print traces）

自己算两种指派，各跑一遍库 CBS，打印矩阵和与 SOC。不要把两列数字比来比去当 bug。


In [ ]:
from evals.scenarios import assign_pair
from lib.algos.assignment import greedy_assign, hungarian_assign, cbs_ta
from lib.algos.cbs import cbs

g, robots, starts, tasks = assign_pair()

hung = hungarian_assign(g, robots, starts, tasks)
greed = greedy_assign(g, robots, starts, tasks)
print("匈牙利 π", hung["assign"], "矩阵和", hung["cost"])
print("贪心   π", greed["assign"], "矩阵和", greed["cost"])

r_h = cbs(g, starts, hung["assign"])
r_g = cbs(g, starts, greed["assign"])
print("匈牙利+CBS  found", r_h["found"], "SOC", r_h["soc"], "paths", r_h["paths"])
print("贪心+CBS    found", r_g["found"], "SOC", r_g["soc"], "paths", r_g["paths"])
print("差值：矩阵和 4、SOC 5 ⇒ 有一拍等待（看路径里重复的点）")


## 8. 逐行实现表

| 行 | 作用 |
|----|------|
| `hungarian_assign` / `greedy_assign` | 只产出 π 与矩阵和，不消撞 |
| `cbs(g, starts, π)` | 目标字典必须是车→节点，且单射 |
| `r["soc"]` | 含等待；与 `hung["cost"]` 单位都是边权时间，但不是同一个量 |
| 打印 `paths` | 连续两个相同节点 = 等待 |
| 两种 π 都跑 | n=2 即穷尽完美匹配；本图 SOC 相同 |

同目标实验：两车都去 G0 时 CBS 往往失败或用尽 cap。


In [ ]:
bad = cbs(g, starts, {"r0": "G0", "r1": "G0"})
print("两车同一目标 found", bad["found"], "SOC", bad.get("soc"), "high", bad.get("high"))

r = cbs_ta(g, robots, starts, tasks)
print("库 cbs_ta 指派", r["assign"], "SOC", r["soc"], "矩阵和", r["assign_cost"], "贪心CBS", r["greedy_soc"])


## 9. 常见 bug

1. **拿矩阵和跟 SOC 比大小当对错。** 4 对 5 在本图是等待，不是实现错。
2. **两车同一目标丢给 CBS。** 终点占用冲突，搜索爆炸或失败。指派必须单射。
3. **以为匈牙利+CBS 就是 CBS-TA。** 它只对 **一个** π 最优。完整 CBS-TA 还要换 π。
4. **CBS 的 starts 写成任务点。** 人还在码头。
5. **比较时一个用欧氏矩阵、一个用 Dijkstra 矩阵。** 先统一 C。

## 10. 对照库

`cbs_ta` 先 `hungarian_assign` 再 `cbs`，并附 `greedy_soc`。eval `test_25`：`soc <= greedy_soc`。完整指派树（下一最优匹配）不在本课作业里。


In [ ]:
from lib.studio.widget import PlannerStudio

studio = PlannerStudio()
studio.show_multi(g.to_studio(), r, "CBS-TA")
studio


## 条目小结

| 项目 | 内容 |
|------|------|
| 问题 | 联合选 π 与无碰撞路径，目标 SOC |
| 匈牙利+CBS | 该 π 的路径最优，不必全局 SOC 最优 |
| 两套代价 | 矩阵和（无等待）vs SOC（有等待） |
| 本图 | 矩阵 4，SOC 5，两种 π 相同 |
| n=2 | 两种匹配穷尽 |

## 练习

1. 在打印出的 paths 里标出等待的 (车, 时刻, 节点)。等待发生在 J 还是起点？
2. 用第 3 节那张 1/5/5/2 的表，说明为什么「矩阵最优」可以 SOC 更差。
3. 若强制两车都去 G0，CBS 的 `found` 是什么？指派层应如何拒绝？
